# Transformer Encoder - IMDB Sentiment Analysis 🎦

**Objective:** Build a Transformer Encoder-based classifier to predict whether a movie review from the [IMDB dataset](https://ai.stanford.edu/~amaas/data/sentiment/) expresses a **positive** or **negative** sentiment.

**Pipeline overview:**
1. Load and explore the dataset
2. Tokenize reviews using a pretrained BERT tokenizer and build a PyTorch `Dataset`
3. Define a Transformer Encoder model
4. Train and evaluate the model

| Component | Detail |
|---|---|
| **Tokenizer** | `bert-base-uncased` (30,522 tokens) |
| **Architecture** | Embedding + TransformerEncoderLayer (8 heads) + Linear head |
| **Loss** | CrossEntropyLoss |
| **Optimizer** | Adam (lr = 1e-3) |
| **Epochs** | 5 |

---
## 1. Load and Explore the Dataset

The IMDB dataset contains **50,000 movie reviews** evenly split between positive and negative labels. We load it from a local CSV and inspect the first rows.

In [1]:
import pandas as pd
df = pd.read_csv("IMDB_DATASET.csv")
print(df.head(5))

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


---
## 2. Tokenization and Dataset Class

We use the **pretrained BERT tokenizer** (`bert-base-uncased`) to convert raw text into token IDs. Each review is:
- **Truncated** or **padded** to a fixed length (`MAX_LEN`)
- Encoded as a tensor of integer indices

The `IMDBData` class wraps the tokenized data into a standard PyTorch `Dataset`, mapping the `"positive"` / `"negative"` labels to `1` / `0`.

In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer

class IMDBData(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.max_len = max_len
        encodings = tokenizer(
            dataframe["review"].tolist(),
            truncation=True,
            max_length=max_len,
            padding="max_length",
            return_tensors="pt"
        )
        self.reviews = encodings["input_ids"]
        self.labels = [1 if s == "positive" else 0 for s in dataframe["sentiment"]]

    def __len__(self):
        return len(self.reviews)

    def __getitem__(self, idx):
        return self.reviews[idx], torch.tensor(self.labels[idx], dtype=torch.long)

---
## 3. Model Architecture

The model follows a simple but effective encoder-only pipeline:

```
Input IDs --> Embedding --> TransformerEncoderLayer --> [CLS] token --> Linear --> 2 classes
```

- **Embedding layer:** Maps each token ID to a dense vector of size `d_model=128`.
- **TransformerEncoderLayer:** Single-layer multi-head self-attention with **8 heads**, allowing the model to attend to different parts of the review simultaneously.
- **Classification head:** Takes the representation of the first token (`[CLS]`) and projects it to 2 output logits (positive / negative).

In [5]:
import torch.nn as nn

class CustomLSTMEncoder(nn.Module):
    def __init__(self, tokenizer, embed_dim=128, d_model=128):
        super().__init__()
        self.embedding = nn.Embedding(tokenizer.vocab_size, d_model, padding_idx=tokenizer.pad_token_id)
        self.encoder = nn.TransformerEncoderLayer(d_model, nhead=8, batch_first=True)
        self.fc = nn.Linear(d_model, 2)

    def forward(self, x):
        emb = self.embedding(x)                    # (batch, seq_len, d_model)
        out = self.encoder(emb)                    # (batch, seq_len, d_model)
        out = out[:, 0, :]                         # token [CLS] -> (batch, d_model)
        return self.fc(out)                        # (batch, 2)

---
## 4. Training Loop

Standard PyTorch training step: forward pass, compute loss, backpropagate gradients, and update weights. Progress is logged every 100 batches.

In [6]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        pred = model(X)
        loss = loss_fn(pred, y)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * dataloader.batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

---
## 5. Evaluation Loop

Runs inference on the test set with gradients disabled (`torch.no_grad`). Reports overall **accuracy** and **average loss** across all batches.

In [7]:
def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

---
## 6. Training and Results

**Setup:**
- Sequences are capped at `MAX_LEN = 200` tokens.
- The dataset is split **80/20** into training (40,000) and test (10,000) samples.
- Batches of **64** reviews are fed to the model.

In [ ]:
MAX_LEN = 200

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
tokenizer.save_pretrained('./mi_tokenizer_bert_local')

print(f"Vocabulary: {tokenizer.vocab_size} words")
print(f"Example: {tokenizer.tokenize('This movie is great!')}")

# Train/test split with torch
train_size = int(len(df) * 0.8)
test_size = len(df) - train_size

full_dataset = IMDBData(df, tokenizer, MAX_LEN)
train_dataset, test_dataset = torch.utils.data.random_split(full_dataset, [train_size, test_size])

train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=64)

In [9]:
model = CustomLSTMEncoder(tokenizer, embed_dim=128)

learning_rate = 1e-3
epochs = 5
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 0.689084  [   64/40000]
loss: 0.517819  [ 6464/40000]
loss: 0.522511  [12864/40000]
loss: 0.539088  [19264/40000]
loss: 0.331155  [25664/40000]
loss: 0.432816  [32064/40000]
loss: 0.336112  [38464/40000]
Test Error: 
 Accuracy: 82.5%, Avg loss: 0.380032 

Epoch 2
-------------------------------
loss: 0.258343  [   64/40000]
loss: 0.548428  [ 6464/40000]
loss: 0.342894  [12864/40000]
loss: 0.220018  [19264/40000]
loss: 0.408245  [25664/40000]
loss: 0.301386  [32064/40000]
loss: 0.452880  [38464/40000]
Test Error: 
 Accuracy: 84.3%, Avg loss: 0.368172 

Epoch 3
-------------------------------
loss: 0.339110  [   64/40000]
loss: 0.262654  [ 6464/40000]
loss: 0.391013  [12864/40000]
loss: 0.203385  [19264/40000]
loss: 0.253809  [25664/40000]
loss: 0.383501  [32064/40000]
loss: 0.146605  [38464/40000]
Test Error: 
 Accuracy: 85.8%, Avg loss: 0.329316 

Epoch 4
-------------------------------
loss: 0.151122  [   64/40000]
loss: 0.311541  [ 6464/4

---
## Results Summary

| Epoch | Train Loss (last batch) | Test Accuracy | Test Avg Loss |
|:-----:|:-----------------------:|:-------------:|:-------------:|
| 1 | 0.336 | 82.5% | 0.380 |
| 2 | 0.453 | 84.3% | 0.368 |
| 3 | 0.147 | 85.8% | 0.329 |
| 4 | 0.299 | 85.8% | 0.347 |
| 5 | 0.208 | **85.9%** | **0.343** |

The model reaches **~86% accuracy** after just 5 epochs with a single Transformer encoder layer and no pretrained weights (only the tokenizer vocabulary is borrowed from BERT). This demonstrates the effectiveness of self-attention for capturing long-range dependencies in text, even in a lightweight setup.